# Final Report: Particle Swarm Optimization in Python

This notebook is the narrated final report for the PSO project. It
documents the experimental methodology, loads the saved results, compares
the implemented execution strategies, and closes with a critical discussion
and practical recommendations.

## 1. Project Goal

The goal of the project is to implement a maintainable Particle Swarm
Optimization solution in Python and use it as a laboratory for comparing
different execution strategies while keeping the optimization core shared.

The final codebase focuses on three comparable variants:

- `V0`: sequential baseline
- `V1`: `ThreadPoolExecutor` evaluation
- `V2`: `ProcessPoolExecutor` evaluation

The rest of the report evaluates how these choices affect runtime,
convergence behavior, and experimental reproducibility.

## 2. Experimental Methodology

### 2.1 Shared PSO Core

All variants use the same PSO implementation, the same update equations,
the same boundary handling, and the same topology. Only the fitness
evaluation backend changes. This is essential for a fair comparison because
it isolates timing differences from optimization-quality differences.

### 2.2 Benchmark Protocol

The benchmark suite evaluates four standard objective functions:

- `Sphere`
- `Rosenbrock`
- `Rastrigin`
- `Ackley`

The benchmark dimensions are `2`, `10`, and `30`. The benchmark YAML uses a
fixed iteration budget, so the strategy comparison is not distorted by early
stopping. Tolerance is still recorded as an analysis metric through
`convergence_iteration`, but it does not terminate benchmark runs.

### 2.3 Grid Search Protocol

The grid search explores combinations of:

- inertia `w`
- cognitive coefficient `c1`
- social coefficient `c2`
- swarm size
- iteration budget

The current configuration runs the reduced search space for `V0`, `V1`, and
`V2`, again with fixed iteration budgets and fixed seeds.

### 2.4 Metrics

The comparison relies on six main metrics:

- final best fitness
- AUC of the best-fitness curve
- convergence iteration
- total runtime
- speedup and efficiency relative to `V0`
- accumulated parallel overhead

These metrics let us compare not only final solution quality but also how
quickly the optimizer reaches good solutions and how much execution overhead
is introduced by the parallel backends.

In [ ]:
from itertools import product
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

benchmark_cfg = yaml.safe_load((ROOT / 'configs' / 'benchmark.yaml').read_text())
grid_cfg = yaml.safe_load((ROOT / 'configs' / 'grid_search.yaml').read_text())

variants = list(benchmark_cfg['strategies'].keys())
objectives = [item['name'] for item in benchmark_cfg['objectives']]
dimensions = benchmark_cfg['dimensions']
benchmark_seeds = benchmark_cfg['seeds']

cases = pd.DataFrame(
    product(variants, objectives, dimensions, benchmark_seeds),
    columns=['variant', 'objective', 'dimensions', 'seed'],
)
cases.head()

### 2.5 Cartesian Product Of Benchmark Cases

In [ ]:
cases.shape, cases.head(12)

## 3. Loading Saved Artifacts

In [ ]:
def load_json(path: Path):
    return json.loads(path.read_text())

def resolve_run_dir(run_dir: str) -> Path:
    run_path = Path(run_dir)
    return run_path if run_path.is_absolute() else ROOT / run_path

benchmark_summary_path = ROOT / 'results' / 'benchmarks' / 'benchmark_summary.csv'
benchmark_runs_path = ROOT / 'results' / 'benchmarks' / 'benchmark_runs.csv'
grid_summary_path = ROOT / 'results' / 'grid_search' / 'grid_search_summary.csv'

benchmark_summary = pd.read_csv(benchmark_summary_path) if benchmark_summary_path.exists() else pd.DataFrame()
benchmark_runs = pd.read_csv(benchmark_runs_path) if benchmark_runs_path.exists() else pd.DataFrame()
grid_summary = pd.read_csv(grid_summary_path) if grid_summary_path.exists() else pd.DataFrame()

benchmark_summary.head()

### 3.1 Recorded Execution Conditions

To avoid misleading micro-benchmarks, the project stores system information
and commit metadata in each saved run. The cell below shows one representative
example from the saved benchmark outputs.

In [ ]:
sample_summary = None
if not benchmark_runs.empty:
    sample_run_dir = resolve_run_dir(benchmark_runs.iloc[0]['run_dir'])
    sample_summary = load_json(sample_run_dir / 'summary.json')
    display({
        'run_id': sample_summary['run_id'],
        'git_commit': sample_summary['git']['commit'],
        'system': sample_summary['system'],
    })
else:
    print('No benchmark runs are available yet.')

## 4. Benchmark Results

The benchmark summary is the first high-level view of the comparison. Since
the PSO core is shared, similar final-fitness values across strategies are
expected. The most important differences should appear in runtime and
parallel overhead.

In [ ]:
if not benchmark_summary.empty:
    display(benchmark_summary.sort_values(['objective', 'dimensions', 'variant']))
else:
    print('No benchmark summary is available yet.')

### 4.1 Best Variant By Runtime

In [ ]:
if not benchmark_summary.empty:
    runtime_winners = (
        benchmark_summary.sort_values('mean_total_time')
        .groupby(['objective', 'dimensions'], as_index=False)
        .first()[['objective', 'dimensions', 'variant', 'mean_total_time']]
        .rename(columns={'variant': 'winner_variant'})
    )
    display(runtime_winners)
else:
    print('No benchmark summary is available yet.')

## 5. Convergence Analysis

Mean convergence curves help compare not only the final solution quality but
also how the optimizer approaches good solutions over time.

In [ ]:
def load_history(run_dir: str) -> pd.DataFrame:
    return pd.read_csv(resolve_run_dir(run_dir) / 'history.csv')

if not benchmark_runs.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    for (variant, objective, dimensions), bucket in benchmark_runs.groupby(['variant', 'objective', 'dimensions']):
        histories = []
        for run_dir in bucket['run_dir']:
            history = load_history(run_dir)
            histories.append(history['best_fitness'].to_numpy())
        max_len = max(len(curve) for curve in histories)
        padded = []
        for curve in histories:
            if len(curve) < max_len:
                pad = [curve[-1]] * (max_len - len(curve))
                curve = list(curve) + pad
            padded.append(curve)
        mean_curve = pd.DataFrame(padded).mean(axis=0)
        ax.plot(mean_curve, label=f'{variant} | {objective} | d={dimensions}', alpha=0.7)
    ax.set_yscale('log')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Best fitness')
    ax.set_title('Mean convergence by case')
    ax.grid(alpha=0.3)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print('No benchmark run data is available yet.')

## 6. Final Fitness Distribution

Boxplots provide a compact visual summary of the distribution of final best
fitness values across strategies.

In [ ]:
if not benchmark_runs.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    benchmark_runs.boxplot(column='best_value', by='variant', ax=ax)
    ax.set_yscale('log')
    ax.set_title('Final fitness by variant')
    ax.set_xlabel('Variant')
    ax.set_ylabel('Final best fitness')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()
else:
    print('No benchmark run data is available yet.')

## 7. Runtime, Speedup, And Efficiency

The most important difference between variants in this project is runtime.
The next table reports speedup relative to `V0` and a simple parallel
efficiency estimate defined as `speedup / workers`.

In [ ]:
if not benchmark_runs.empty:
    worker_map = {'V0': 1}
    for variant, overrides in benchmark_cfg['strategies'].items():
        worker_map[variant] = overrides.get('workers', 1)

    speedup_rows = []
    for dimensions, bucket in benchmark_runs.groupby('dimensions'):
        baseline = bucket[bucket['variant'] == 'V0']['total_time'].mean()
        for variant, variant_bucket in bucket.groupby('variant'):
            mean_time = variant_bucket['total_time'].mean()
            speedup = baseline / mean_time
            workers = worker_map.get(variant, 1)
            efficiency = speedup / workers if workers else None
            speedup_rows.append({
                'dimensions': dimensions,
                'variant': variant,
                'workers': workers,
                'mean_total_time': mean_time,
                'speedup_vs_V0': speedup,
                'efficiency': efficiency,
            })
    speedup_df = pd.DataFrame(speedup_rows)
    display(speedup_df.sort_values(['dimensions', 'variant']))
else:
    print('No benchmark run data is available yet.')

## 8. Overhead Analysis

Parallelization is only useful when the extra coordination cost remains under
control. The next cell reads the per-run summaries and compares the recorded
total overhead accumulated by each strategy.

In [ ]:
if not benchmark_runs.empty:
    overhead_rows = []
    for _, row in benchmark_runs.iterrows():
        summary = load_json(resolve_run_dir(row['run_dir']) / 'summary.json')
        overhead_rows.append({
            'variant': row['variant'],
            'objective': row['objective'],
            'dimensions': row['dimensions'],
            'total_overhead_time': summary['metrics']['total_overhead_time'],
        })
    overhead_df = pd.DataFrame(overhead_rows)
    overhead_summary = (
        overhead_df.groupby(['variant', 'dimensions'], as_index=False)['total_overhead_time']
        .mean()
        .rename(columns={'total_overhead_time': 'mean_total_overhead_time'})
    )
    display(overhead_summary.sort_values(['dimensions', 'variant']))
else:
    print('No benchmark run data is available yet.')

## 9. Grid Search Results

The grid search ranks hyperparameter combinations according to the selected
metric. In the current configuration, the reduced search space is evaluated
for each strategy so that the search procedure is comparable across `V0`,
`V1`, and `V2`.

In [ ]:
if not grid_summary.empty:
    display(grid_summary.sort_values(['variant', 'mean_best_value']).head(15))
else:
    print('No grid-search summary is available yet.')

## 10. Critical Discussion

### 10.1 GIL And Threading

The thread-based variant is useful for demonstrating that Python threads do
not automatically improve CPU-bound numerical workloads. When the objective
evaluation is dominated by Python-side work, the Global Interpreter Lock can
limit parallel speedup.

### 10.2 IPC And Process Overhead

The process-based variant avoids the GIL for the worker side, but it pays
extra cost in serialization, process startup, synchronization, and inter-
process communication. This overhead matters especially when each task is
too small.

### 10.3 Why Fitness Quality Stays Similar

Because the PSO core is shared and the order of particle results is preserved
by the evaluators, optimization quality remains the same or nearly the same
across `V0`, `V1`, and `V2`. This is expected in the current architecture and
is actually desirable for a fair comparison.

### 10.4 Vectorization As A Discussion Point

Vectorization is not part of the final simplified codebase, but it remains an
important discussion point. In many numerical optimization tasks, vectorized
NumPy code can outperform both threads and processes because it reduces Python
loop overhead and delegates work to optimized native routines. It is therefore
a relevant comparison point in the report even if it is not part of the final
mandatory implementation.

### 10.5 Trade-Off Summary

- `V0` is simple, stable, and often best when the workload is small.
- `V1` is pedagogically useful for discussing the GIL, but it is not always faster.
- `V2` is the most promising for heavier workloads, but only when the evaluation cost is large enough to amortize IPC overhead.

## 11. Recommendations

Based on the current implementation and saved benchmark results, the main
practical recommendations are:

- Use `V0` as the default baseline for lightweight problems and debugging.
- Use `V2` when the objective evaluation is expensive enough to justify process overhead.
- Keep benchmark and grid-search runs on fixed iteration budgets so timing comparisons remain fair.
- Record seeds, configuration, and system metadata in every run so experiments stay reproducible.
- Treat vectorization as an important future direction even if it is outside the current simplified scope.

Overall, the codebase behaves as a maintainable PSO laboratory with a shared
optimizer core, structured persistence, reproducible experiments, and clear
separation between optimization logic and evaluation strategy.